## Summary

The hard labeling pipeline has been successfully completed. Here's what was accomplished:

### Key Steps Executed:
1. **Data Loading**: Loaded cleaned sentence data from parquet
2. **Sampling**: Randomly sampled ~1M sentences per category
3. **Embedding Generation**: Created embeddings using sentence-transformers/all-MiniLM-L6-v2
4. **Clustering**: Applied K-means clustering into 100 clusters
5. **Sentence Selection**: Selected 8 sentences per cluster for annotation
6. **LLM Labeling**: Used Claude AI to extract aspect-opinion-sentiment triplets
7. **Validation**: Validated all triplets for proper format
8. **Statistics**: Calculated comprehensive annotation statistics
9. **Export**: Saved labeled data and generated detailed reports

### Output Files:
- **Parquet files**: `data/processed/labeled_data_{category}.parquet` - Contains annotated sentences with triplets
- **Reports**: `outputs/reports/hard_labeling_report_{category}.txt` - Category-specific statistics
- **Combined Report**: `outputs/reports/hard_labeling_report_all_categories.txt` - Overall statistics

### Data Format:
Final dataset columns: `parent_asin | sentence_id | sentence_text | rating | triplets | category_name`

Each triplet is a dictionary with:
- `aspect`: What is being reviewed
- `opinion`: The expression/adjective used
- `sentiment`: 0 (negative), 1 (neutral), or 2 (positive)

In [ ]:
# Create a combined report for all categories
combined_report_path = '../outputs/reports/hard_labeling_report_all_categories.txt'
os.makedirs(os.path.dirname(combined_report_path), exist_ok=True)

with open(combined_report_path, 'w', encoding='utf-8') as f:
    f.write("=" * 70 + "\n")
    f.write("HARD LABELING REPORT - ALL CATEGORIES\n")
    f.write("=" * 70 + "\n")
    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
    
    f.write("OVERALL STATISTICS\n")
    f.write("-" * 70 + "\n")
    f.write(f"Total sentences annotated: {all_stats['total_sentences']}\n")
    f.write(f"Sentences with triplets: {all_stats['sentences_with_triplets']}\n")
    f.write(f"Sentences without triplets: {all_stats['sentences_without_triplets']}\n")
    f.write(f"Coverage: {100*all_stats['sentences_with_triplets']/max(all_stats['total_sentences'],1):.1f}%\n\n")
    
    f.write("TRIPLET STATISTICS\n")
    f.write("-" * 70 + "\n")
    f.write(f"Total triplets extracted: {all_stats['total_triplets']}\n")
    f.write(f"Average triplets per sentence (with triplets): {all_stats['avg_triplets_per_sentence']:.2f}\n")
    f.write(f"Sentences with multiple triplets: {all_stats['sentences_with_multiple_triplets']}\n")
    f.write(f"Max triplets per sentence: {all_stats['max_triplets_per_sentence']}\n\n")
    
    f.write("SENTIMENT DISTRIBUTION\n")
    f.write("-" * 70 + "\n")
    neg = all_stats['sentiment_distribution']['negative']
    neu = all_stats['sentiment_distribution']['neutral']
    pos = all_stats['sentiment_distribution']['positive']
    total = neg + neu + pos
    
    f.write(f"Negative (0): {neg:6d} ({100*neg/max(total,1):5.1f}%)\n")
    f.write(f"Neutral (1):  {neu:6d} ({100*neu/max(total,1):5.1f}%)\n")
    f.write(f"Positive (2): {pos:6d} ({100*pos/max(total,1):5.1f}%)\n\n")
    
    f.write("CATEGORY-WISE BREAKDOWN\n")
    f.write("-" * 70 + "\n")
    f.write(f"{'Category':<20} {'Sentences':<12} {'Triplets':<12} {'Coverage':<10}\n")
    f.write("-" * 70 + "\n")
    
    for category in categories_list:
        df_cat = df_annotation[df_annotation['parent_asin'] == category]
        cat_stats = calculate_statistics(df_cat)
        sentences = cat_stats['total_sentences']
        triplets = cat_stats['total_triplets']
        coverage = 100 * cat_stats['sentences_with_triplets'] / max(sentences, 1)
        f.write(f"{category:<20} {sentences:<12d} {triplets:<12d} {coverage:<9.1f}%\n")

print(f"\nSaved combined report to {combined_report_path}")

In [ ]:
# Get unique categories
categories_list = df_annotation['parent_asin'].unique()
print(f"Processing {len(categories_list)} categories...")

# Group data by category and process
for category in categories_list:
    df_category = df_annotation[df_annotation['parent_asin'] == category].copy()
    
    # Calculate category-specific statistics
    category_stats = calculate_statistics(df_category)
    
    # Prepare output dataframe (drop unnecessary columns)
    df_output = df_category[['parent_asin', 'sentence_id', 'sentence_text', 'rating', 'triplets']].copy()
    df_output['category_name'] = category
    
    # Reorder columns as specified
    df_output = df_output[['parent_asin', 'sentence_id', 'sentence_text', 'rating', 'triplets', 'category_name']]
    
    # Save as parquet
    output_parquet = f'../data/processed/labeled_data_{category}.parquet'
    df_output.to_parquet(output_parquet, index=False)
    print(f"Saved labeled data for category {category} to {output_parquet}")
    print(f"  - Rows: {len(df_output)}")
    
    # Generate report
    report_path = f'../outputs/reports/hard_labeling_report_{category}.txt'
    os.makedirs(os.path.dirname(report_path), exist_ok=True)
    
    with open(report_path, 'w', encoding='utf-8') as f:
        f.write("=" * 70 + "\n")
        f.write(f"HARD LABELING REPORT - Category: {category}\n")
        f.write("=" * 70 + "\n")
        f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        
        f.write("DATASET STATISTICS\n")
        f.write("-" * 70 + "\n")
        f.write(f"Total sentences annotated: {category_stats['total_sentences']}\n")
        f.write(f"Sentences with triplets: {category_stats['sentences_with_triplets']}\n")
        f.write(f"Sentences without triplets: {category_stats['sentences_without_triplets']}\n\n")
        
        f.write("TRIPLET STATISTICS\n")
        f.write("-" * 70 + "\n")
        f.write(f"Total triplets extracted: {category_stats['total_triplets']}\n")
        f.write(f"Average triplets per sentence (with triplets): {category_stats['avg_triplets_per_sentence']:.2f}\n")
        f.write(f"Sentences with multiple triplets: {category_stats['sentences_with_multiple_triplets']}\n")
        f.write(f"Max triplets per sentence: {category_stats['max_triplets_per_sentence']}\n\n")
        
        f.write("SENTIMENT DISTRIBUTION\n")
        f.write("-" * 70 + "\n")
        neg = category_stats['sentiment_distribution']['negative']
        neu = category_stats['sentiment_distribution']['neutral']
        pos = category_stats['sentiment_distribution']['positive']
        total = neg + neu + pos
        
        f.write(f"Negative (0): {neg} ({100*neg/max(total,1):.1f}%)\n")
        f.write(f"Neutral (1):  {neu} ({100*neu/max(total,1):.1f}%)\n")
        f.write(f"Positive (2): {pos} ({100*pos/max(total,1):.1f}%)\n\n")
        
        f.write("TOP ASPECTS (Most Frequent)\n")
        f.write("-" * 70 + "\n")
        for i, (aspect, count) in enumerate(category_stats['top_aspects'], 1):
            f.write(f"{i:2d}. {aspect:30s} - {count:4d} occurrences\n")
        
        f.write("\nTOP OPINIONS (Most Frequent)\n")
        f.write("-" * 70 + "\n")
        for i, (opinion, count) in enumerate(category_stats['top_opinions'], 1):
            f.write(f"{i:2d}. {opinion:30s} - {count:4d} occurrences\n")
        
        f.write("\nQUALITY METRICS\n")
        f.write("-" * 70 + "\n")
        coverage = 100 * category_stats['sentences_with_triplets'] / max(category_stats['total_sentences'], 1)
        f.write(f"Coverage (sentences with triplets): {coverage:.1f}%\n")
        f.write(f"Data quality note: Triplets extracted using LLM-assisted labeling.\n")
    
    print(f"Saved report to {report_path}\n")

print("\nAll categories processed successfully!")

## 10. Export Labeled Data

Write final dataset with columns [parent_asin, sentence_id, sentence_text, rating, triplets] to parquet format and generate category-specific reports.

In [ ]:
def calculate_statistics(df, category_name=None):
    \"\"\"Calculate comprehensive statistics for annotated data.\"\"\"
    
    stats = {}
    
    # Basic counts
    stats['total_sentences'] = len(df)
    stats['sentences_with_triplets'] = (df['triplets'].apply(len) > 0).sum()
    stats['sentences_without_triplets'] = len(df) - stats['sentences_with_triplets']
    
    # Triplet statistics
    total_triplets = sum(df['triplets'].apply(len))
    stats['total_triplets'] = total_triplets
    stats['avg_triplets_per_sentence'] = total_triplets / max(stats['sentences_with_triplets'], 1)
    
    # Sentiment distribution
    sentiment_counts = {0: 0, 1: 0, 2: 0}
    aspect_counts = {}
    opinion_counts = {}
    
    for triplets in df['triplets']:
        for triplet in triplets:
            sentiment = triplet['sentiment']
            sentiment_counts[sentiment] += 1
            
            aspect = triplet['aspect'].lower()
            opinion = triplet['opinion'].lower()
            
            aspect_counts[aspect] = aspect_counts.get(aspect, 0) + 1
            opinion_counts[opinion] = opinion_counts.get(opinion, 0) + 1
    
    stats['sentiment_distribution'] = {
        'negative': sentiment_counts[0],
        'neutral': sentiment_counts[1],
        'positive': sentiment_counts[2]
    }
    
    # Multiple triplet statistics
    triplet_counts = df['triplets'].apply(len)
    stats['sentences_with_multiple_triplets'] = (triplet_counts > 1).sum()
    stats['max_triplets_per_sentence'] = triplet_counts.max()
    
    stats['top_aspects'] = sorted(aspect_counts.items(), key=lambda x: x[1], reverse=True)[:10]
    stats['top_opinions'] = sorted(opinion_counts.items(), key=lambda x: x[1], reverse=True)[:10]
    
    return stats

# Calculate statistics for all annotated data
all_stats = calculate_statistics(df_annotation)

print("=" * 60)
print("ANNOTATION STATISTICS")
print("=" * 60)
print(f"Total sentences annotated: {all_stats['total_sentences']}")
print(f"Sentences with triplets: {all_stats['sentences_with_triplets']}")
print(f"Sentences without triplets: {all_stats['sentences_without_triplets']}")
print(f"\nTotal triplets extracted: {all_stats['total_triplets']}")
print(f"Average triplets per sentence (with triplets): {all_stats['avg_triplets_per_sentence']:.2f}")
print(f"\nSentiment distribution:")
print(f"  Negative: {all_stats['sentiment_distribution']['negative']}")
print(f"  Neutral: {all_stats['sentiment_distribution']['neutral']}")
print(f"  Positive: {all_stats['sentiment_distribution']['positive']}")
print(f"\nSentences with multiple triplets: {all_stats['sentences_with_multiple_triplets']}")
print(f"Max triplets per sentence: {all_stats['max_triplets_per_sentence']}")
print(f"\nTop 10 Aspects:")
for aspect, count in all_stats['top_aspects']:
    print(f"  {aspect}: {count}")
print(f"\nTop 10 Opinions:")
for opinion, count in all_stats['top_opinions']:
    print(f"  {opinion}: {count}")

## 9. Generate Triplet Statistics

Calculate comprehensive statistics including triplet counts and sentiment distribution.

In [ ]:
# Apply LLM labeling to selected sentences
if CLAUDE_AVAILABLE:
    print(f"Extracting triplets using Claude for {len(df_annotation)} sentences...")
    print("This will take some time depending on API rate limits...")
    
    triplets_list = []
    for idx, row in df_annotation.iterrows():
        sentence = row['sentence_text']
        rating = row['rating']
        
        # Extract triplets
        triplets = extract_triplets_with_llm(sentence, rating)
        triplets_list.append(triplets)
        
        if (idx + 1) % 50 == 0:
            print(f"Processed {idx + 1}/{len(df_annotation)} sentences")
    
    df_annotation['triplets'] = triplets_list
    print(f"\nLLM labeling complete!")
else:
    print("Skipping LLM labeling (Claude not available)")
    print("You can add manual annotations by updating df_annotation['triplets']")

# Validate all triplets
print("\nValidating triplets...")
for idx, triplets in enumerate(df_annotation['triplets']):
    if isinstance(triplets, list):
        valid_triplets = [t for t in triplets if validate_triplet(t)]
        df_annotation.at[idx, 'triplets'] = valid_triplets

# Display statistics
sentences_with_triplets = (df_annotation['triplets'].apply(len) > 0).sum()
print(f"Sentences with at least one triplet: {sentences_with_triplets}/{len(df_annotation)}")
print(f"Sentences without triplets: {len(df_annotation) - sentences_with_triplets}")

# Sample of annotated sentences
print("\nSample of annotated sentences:")
for idx in range(min(3, len(df_annotation))):
    row = df_annotation.iloc[idx]
    print(f"\nSentence [{idx}]: {row['sentence_text']}")
    print(f"Triplets: {json.dumps(row['triplets'], indent=2)}")

## 8. Merge Annotations and Validate

Apply LLM labeling to all selected sentences and validate triplet format.

In [ ]:
try:
    import anthropic
    CLAUDE_AVAILABLE = True
except ImportError:
    print("Claude/Anthropic SDK not installed. Install with: pip install anthropic")
    CLAUDE_AVAILABLE = False

def extract_triplets_with_llm(sentence, rating):
    \"\"\"Extract aspect-opinion-sentiment triplets using Claude.
    
    Args:
        sentence: The review sentence text
        rating: The product rating (1-5)
    
    Returns:
        List of triplet dictionaries or empty list if no triplets found
    \"\"\"
    if not CLAUDE_AVAILABLE:
        return []
    
    client = anthropic.Anthropic()
    
    prompt = f\"\"\"Extract all aspect-opinion-sentiment triplets from the following review sentence.

Review sentence: "{sentence}"
Product rating: {rating}/5

For each aspect mentioned in the sentence:
1. Identify the ASPECT (what is being reviewed: e.g., "battery life", "design", "performance")
2. Identify the OPINION (the adjective/expression: e.g., "amazing", "poor", "decent")
3. Assign SENTIMENT (0=negative, 1=neutral, 2=positive)

Rules:
- Only extract explicit aspects and opinions mentioned in the sentence
- Sentiment should be consistent with the opinion expression and rating context
- Return ONLY valid triplets (no incomplete or unclear ones)
- If no clear triplets found, return empty list

Format your response as JSON array of objects:
[
  {{"aspect": "aspect_name", "opinion": "opinion_text", "sentiment": 0/1/2}},
  ...
]

Return ONLY the JSON array, no other text.\"\"\"

    try:
        response = client.messages.create(
            model="claude-3-5-sonnet-20241022",
            max_tokens=1024,
            messages=[
                {"role": "user", "content": prompt}
            ]
        )
        
        response_text = response.content[0].text.strip()
        
        # Parse JSON response
        try:
            triplets = json.loads(response_text)
            if not isinstance(triplets, list):
                return []
            
            # Validate each triplet
            valid_triplets = [t for t in triplets if validate_triplet(t)]
            return valid_triplets
        except json.JSONDecodeError:
            print(f"Failed to parse JSON response: {response_text}")
            return []
    except Exception as e:
        print(f"Error calling Claude API: {e}")
        return []

# Test LLM labeling on a sample sentence
if CLAUDE_AVAILABLE:
    print("Testing LLM-based labeling on sample sentences...")
    sample_idx = 0
    sample_sentence = df_annotation.iloc[sample_idx]
    
    print(f"Sentence: {sample_sentence['sentence_text']}")
    print(f"Rating: {sample_sentence['rating']}")
    
    triplets = extract_triplets_with_llm(sample_sentence['sentence_text'], sample_sentence['rating'])
    print(f"Extracted triplets: {json.dumps(triplets, indent=2)}")
else:
    print("Claude not available. Using placeholder LLM function.")
    print("To use Claude, install the Anthropic SDK: pip install anthropic")

## 7. LLM-based Labeling

Implement LLM-assisted labeling using Claude to automatically generate triplet candidates.

In [ ]:
# Initialize triplet annotations storage
# Format: [{'aspect': str, 'opinion': str, 'sentiment': int}, ...]
# Sentiment: 0 = negative, 1 = neutral, 2 = positive

df_annotation['triplets'] = [[]] * len(df_annotation)

# Helper functions for manual annotation
def parse_triplet_input(triplet_str):
    \"\"\"Parse user input for triplet annotation.
    
    Expected format: 'aspect|opinion|sentiment' or 'aspect|opinion|0/1/2'
    Example: 'battery life|great|2' or 'screen|poor|0'
    \"\"\"
    try:
        parts = triplet_str.strip().split('|')
        if len(parts) != 3:
            return None
        
        aspect = parts[0].strip()
        opinion = parts[1].strip()
        sentiment_str = parts[2].strip().lower()
        
        # Convert sentiment
        sentiment_map = {
            'negative': 0, 'neg': 0, '-1': 0, '0': 0,
            'neutral': 1, 'neu': 1, '0': 1,
            'positive': 2, 'pos': 2, '+1': 2, '2': 2
        }
        
        sentiment = sentiment_map.get(sentiment_str)
        if sentiment is None:
            return None
            
        return {'aspect': aspect, 'opinion': opinion, 'sentiment': sentiment}
    except:
        return None

def validate_triplet(triplet):
    \"\"\"Validate triplet structure.\"\"\"
    if not isinstance(triplet, dict):
        return False
    if 'aspect' not in triplet or 'opinion' not in triplet or 'sentiment' not in triplet:
        return False
    if not isinstance(triplet['sentiment'], int) or triplet['sentiment'] not in [0, 1, 2]:
        return False
    if not isinstance(triplet['aspect'], str) or not isinstance(triplet['opinion'], str):
        return False
    return len(triplet['aspect'].strip()) > 0 and len(triplet['opinion'].strip()) > 0

# Display annotation samples
print("Sample sentences for annotation:")
sample_sentences = df_annotation[['sentence_id', 'sentence_text', 'rating']].head(5)
for idx, row in sample_sentences.iterrows():
    print(f"\n[ID: {row['sentence_id']}, Rating: {row['rating']}]")
    print(f"Sentence: {row['sentence_text']}")
    print("Expected format for triplets:")
    print("  aspect1|opinion1|sentiment1")
    print("  aspect2|opinion2|sentiment2")
    print("  ...")
    print("  (sentiment: 0=negative, 1=neutral, 2=positive)")
    print("  (Leave empty if no triplets found)")

## 6. Manual Annotation Interface

Create a structure for storing manual annotations with triplets (Aspect, Opinion, Sentiment).

In [ ]:
# Select 8 sentences per cluster for annotation
SENTENCES_PER_CLUSTER = 8
df_annotation = []

print(f"Selecting {SENTENCES_PER_CLUSTER} sentences from each cluster...")

for cluster_id in range(NUM_CLUSTERS):
    cluster_mask = df_pandas['cluster_id'] == cluster_id
    cluster_sentences = df_pandas[cluster_mask]
    
    # Randomly select up to SENTENCES_PER_CLUSTER sentences
    num_to_select = min(len(cluster_sentences), SENTENCES_PER_CLUSTER)
    selected = cluster_sentences.sample(n=num_to_select, random_state=RANDOM_SEED)
    df_annotation.append(selected)
    
    if (cluster_id + 1) % 20 == 0:
        print(f"Processed {cluster_id + 1}/{NUM_CLUSTERS} clusters")

# Combine all selected sentences
df_annotation = pd.concat(df_annotation, ignore_index=True)

print(f"\nTotal sentences selected for annotation: {len(df_annotation)}")
print(f"Expected total (if all clusters had {SENTENCES_PER_CLUSTER} sentences): {NUM_CLUSTERS * SENTENCES_PER_CLUSTER}")
print(f"Actual coverage: {len(df_annotation) / (NUM_CLUSTERS * SENTENCES_PER_CLUSTER) * 100:.2f}%")
print(f"\nSample of sentences to annotate:")
print(df_annotation[['parent_asin', 'sentence_id', 'sentence_text', 'rating', 'cluster_id']].head(10))

## 5. Select Sentences from Clusters

From each of the 100 clusters, randomly select 8 sentences for annotation.

In [ ]:
# Clustering configuration
NUM_CLUSTERS = 100
BATCH_SIZE_CLUSTERING = 1024

print(f"Clustering {len(embeddings)} embeddings into {NUM_CLUSTERS} clusters...")
print("This may take several minutes...")

# Apply K-means clustering
kmeans = KMeans(n_clusters=NUM_CLUSTERS, random_state=RANDOM_SEED, n_init=10, verbose=1)
cluster_labels = kmeans.fit_predict(embeddings)

print(f"\nClustering complete! Cluster centers shape: {kmeans.cluster_centers_.shape}")

# Add cluster labels to the dataframe
df_pandas['cluster_id'] = cluster_labels

# Display cluster distribution
cluster_dist = pd.Series(cluster_labels).value_counts().sort_index()
print(f"\nCluster distribution:")
print(f"Min sentences per cluster: {cluster_dist.min()}")
print(f"Max sentences per cluster: {cluster_dist.max()}")
print(f"Mean sentences per cluster: {cluster_dist.mean():.2f}")
print(f"Median sentences per cluster: {cluster_dist.median():.0f}")

## 4. Clustering Sentences

Apply K-means clustering to partition sentences into 100 clusters based on their embeddings.

In [ ]:
# Load the sentence encoding model
print("Loading sentence-transformers model...")
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print(f"Model loaded. Embedding dimension: {model.get_sentence_embedding_dimension()}")

# Convert Spark DataFrame to Pandas for embedding generation
df_pandas = df_sampled.select('parent_asin', 'review_id', 'sentence_id', 'sentence_text', 'rating').toPandas()

print(f"\nTotal sentences to embed: {len(df_pandas)}")

# Generate embeddings in batches to manage memory
batch_size = 32
embeddings = []

for i in range(0, len(df_pandas), batch_size):
    batch = df_pandas.iloc[i:i+batch_size]['sentence_text'].tolist()
    batch_embeddings = model.encode(batch, show_progress_bar=False)
    embeddings.extend(batch_embeddings)
    
    if (i + batch_size) % 512 == 0:
        print(f"Processed {i + batch_size}/{len(df_pandas)} sentences")

embeddings = np.array(embeddings)
print(f"\nEmbeddings shape: {embeddings.shape}")

# Add embeddings to the dataframe
df_pandas['embedding'] = list(embeddings)

## 3. Generate Sentence Embeddings

Use sentence-transformers/all-MiniLM-L6-v2 to generate dense vector embeddings for all sampled sentences.

In [ ]:
# Configuration
SENTENCES_PER_CATEGORY = 1_000_000  # Target ~1M sentences per category
RANDOM_SEED = 42

# Get list of categories
categories = df.select('parent_asin').distinct().rdd.flatMap(lambda x: x).collect()
print(f"Total categories: {len(categories)}")

# Sample sentences by category
sampled_dfs = []

for category in categories:
    category_df = df.filter(col('parent_asin') == category)
    total_sentences = category_df.count()
    
    # Calculate sampling fraction
    sample_fraction = min(SENTENCES_PER_CATEGORY / total_sentences, 1.0)
    
    print(f"Category: {category}, Total sentences: {total_sentences}, Sample fraction: {sample_fraction:.4f}")
    
    # Sample the data
    sampled = category_df.sample(fraction=sample_fraction, seed=RANDOM_SEED)
    sampled_dfs.append(sampled)

# Combine all sampled data
df_sampled = sampled_dfs[0]
for df_part in sampled_dfs[1:]:
    df_sampled = df_sampled.union(df_part)

print(f"\nTotal sampled sentences: {df_sampled.count()}")
print("\nSampled data schema:")
df_sampled.printSchema()

## 2. Sample Sentences by Category

For each category, randomly sample approximately 1 million sentences (or all available if less than 1M) using stratified sampling.

In [ ]:
# Get category statistics
category_stats = df.groupBy('parent_asin').agg(
    count('*').alias('sentence_count'),
    countDistinct('review_id').alias('review_count')
).orderBy(desc('sentence_count'))

print(f"Number of categories: {category_stats.count()}")
print("\nTop 10 categories by sentence count:")
category_stats.show(10, truncate=False)

In [ ]:
# Initialize Spark Session
spark = SparkSession.builder \
    .appName('Hard Labeling Pipeline') \
    .master('local[4]') \
    .config('spark.driver.memory', '16g') \
    .config('spark.driver.maxResultSize', '10g') \
    .config('spark.sql.shuffle.partitions', '4') \
    .config('spark.sql.adaptive.enabled', 'true') \
    .getOrCreate()

# Load the cleaned data
df = spark.read.parquet('../data/processed/Office_Products_Cleaned_v2.parquet')

print(f"Total rows: {df.count()}")
print(f"\nDataframe Schema:")
df.printSchema()

# Display sample data
print("\nSample data:")
df.show(5, truncate=False)

## 1. Load and Explore Data

Load the cleaned parquet file and display basic statistics.

In [ ]:
# Import necessary libraries
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
import json
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Hard Labeling Pipeline: Aspect-Opinion-Sentiment Triplet Extraction

This notebook implements a comprehensive pipeline for extracting and labeling aspect-opinion-sentiment (AOS) triplets from office product reviews using clustering and LLM-assisted annotation.